In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Live_Query_Engine_Test") \
    .master("local[2]") \
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,"
            "org.apache.hadoop:hadoop-aws:3.4.1,"
            "software.amazon.awssdk:bundle:2.29.38") \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local",           "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type",      "rest") \
    .config("spark.sql.catalog.local.uri",       "http://rest-catalog:8181") \
    .config("spark.sql.catalog.local.warehouse", "s3a://business-data/") \
    .config("spark.sql.catalog.local.s3.endpoint",          "http://minio:9000") \
    .config("spark.sql.catalog.local.s3.path-style-access", "true") \
    .config("spark.sql.catalog.local.s3.region",            "us-east-1") \
    .config("spark.hadoop.fs.s3a.endpoint",          "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",        "admin") \
    .config("spark.hadoop.fs.s3a.secret.key",        "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Session Ready! Version: {spark.version}")

Spark Session Ready! Version: 4.0.2


In [2]:
# Query the live Iceberg table
spark.sql("""
    SELECT 
        traffic_event_ts, 
        traffic_borough, 
        traffic_speed, 
        aq_pm25_ugm3, 
        weather_temperature
    FROM local.db.enriched_traffic
    ORDER BY traffic_event_ts DESC
    LIMIT 10
""").show(truncate=False)

+-------------------+---------------+-------------+------------+-------------------+
|traffic_event_ts   |traffic_borough|traffic_speed|aq_pm25_ugm3|weather_temperature|
+-------------------+---------------+-------------+------------+-------------------+
|2026-05-06 22:43:13|Staten Island  |41.63        |20.3        |65                 |
|2026-05-06 22:43:13|Staten Island  |41.63        |20.3        |66                 |
|2026-05-06 22:43:13|Staten Island  |41.63        |20.3        |65                 |
|2026-05-06 22:43:13|Staten Island  |41.63        |20.3        |60                 |
|2026-05-06 22:43:13|Staten Island  |41.63        |20.3        |63                 |
|2026-05-06 22:43:13|Staten Island  |41.63        |20.3        |66                 |
|2026-05-06 22:43:13|Staten Island  |41.63        |20.3        |65                 |
|2026-05-06 22:43:13|Staten Island  |41.63        |20.3        |65                 |
|2026-05-06 22:43:13|Staten Island  |41.63        |20.3        |6

In [3]:
# Verify the aggregations and non-null air quality readings
spark.sql("""
    SELECT 
        traffic_borough, 
        COUNT(*) as total_records,
        ROUND(AVG(traffic_speed), 2) as avg_speed_mph, 
        ROUND(AVG(aq_pm25_ugm3), 2) as avg_pm25,
        MAX(weather_temperature) as current_temp
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough
    ORDER BY avg_speed_mph ASC
""").show(truncate=False)

+---------------+-------------+-------------+--------+------------+
|traffic_borough|total_records|avg_speed_mph|avg_pm25|current_temp|
+---------------+-------------+-------------+--------+------------+
|Manhattan      |588230       |9.71         |17.75   |66          |
|Bronx          |308725       |11.72        |17.6    |66          |
|Queens         |104780       |12.6         |1.38    |66          |
|Staten Island  |60840        |32.05        |19.94   |66          |
|Brooklyn       |425880       |45.97        |9.72    |66          |
+---------------+-------------+-------------+--------+------------+



In [4]:
# 1. Read the raw Parquet dumps directly from MinIO
raw_traffic = spark.read.parquet("s3a://raw-data/traffic/")
raw_traffic.createOrReplaceTempView("raw_traffic_view")

# 2. See how many records have successfully landed
spark.sql("""
    SELECT borough, COUNT(*) as rows_ingested 
    FROM raw_traffic_view 
    GROUP BY borough
""").show()

+-------------+-------------+
|      borough|rows_ingested|
+-------------+-------------+
|       Queens|          666|
|     Brooklyn|          216|
|Staten Island|          468|
|    Manhattan|          468|
|        Bronx|          432|
+-------------+-------------+



In [5]:
# Borough-level diagnostics: traffic-only vs traffic+AQ vs traffic+AQ+weather
spark.sql("""
WITH traffic_only AS (
  SELECT borough AS traffic_borough, COUNT(*) AS traffic_rows
  FROM parquet.`s3a://raw-data/traffic/`
  WHERE borough IS NOT NULL
  GROUP BY borough
),
aq_joined AS (
  SELECT traffic_borough, COUNT(*) AS traffic_aq_rows
  FROM local.db.enriched_traffic
  GROUP BY traffic_borough
),
aq_weather_joined AS (
  SELECT traffic_borough, COUNT(*) AS traffic_aq_weather_rows
  FROM local.db.enriched_traffic
  WHERE weather_temperature IS NOT NULL
  GROUP BY traffic_borough
)
SELECT
  t.traffic_borough,
  t.traffic_rows,
  COALESCE(a.traffic_aq_rows, 0) AS traffic_aq_rows,
  COALESCE(w.traffic_aq_weather_rows, 0) AS traffic_aq_weather_rows,
  ROUND(100.0 * COALESCE(a.traffic_aq_rows, 0) / t.traffic_rows, 2) AS aq_match_pct,
  ROUND(100.0 * COALESCE(w.traffic_aq_weather_rows, 0) / t.traffic_rows, 2) AS aq_weather_match_pct
FROM traffic_only t
LEFT JOIN aq_joined a ON t.traffic_borough = a.traffic_borough
LEFT JOIN aq_weather_joined w ON t.traffic_borough = w.traffic_borough
ORDER BY t.traffic_rows DESC
""").show(truncate=False)

+---------------+------------+---------------+-----------------------+------------+--------------------+
|traffic_borough|traffic_rows|traffic_aq_rows|traffic_aq_weather_rows|aq_match_pct|aq_weather_match_pct|
+---------------+------------+---------------+-----------------------+------------+--------------------+
|Queens         |666         |104780         |104780                 |15732.73    |15732.73            |
|Staten Island  |468         |60840          |60840                  |13000.00    |13000.00            |
|Manhattan      |468         |588230         |588230                 |125690.17   |125690.17           |
|Bronx          |432         |308725         |308725                 |71464.12    |71464.12            |
|Brooklyn       |216         |425880         |425880                 |197166.67   |197166.67           |
+---------------+------------+---------------+-----------------------+------------+--------------------+



In [6]:
spark.read.parquet("s3a://raw-data/weather/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

spark.read.parquet("s3a://raw-data/openaq/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-05-06 22:58:50.334|2026-05-06 23:39:37.837|
+-----------------------+-----------------------+

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-05-06 22:58:50.598|2026-05-06 23:41:26.404|
+-----------------------+-----------------------+



In [7]:
spark.sql("""
    SELECT MIN(traffic_event_ts), MAX(traffic_event_ts), COUNT(*) as total
    FROM local.db.enriched_traffic
""").show(truncate=False)

+---------------------+---------------------+-------+
|min(traffic_event_ts)|max(traffic_event_ts)|total  |
+---------------------+---------------------+-------+
|2026-05-06 21:18:00  |2026-05-06 22:43:13  |1488455|
+---------------------+---------------------+-------+



In [8]:
spark.sql("""
    SELECT 
        MAX(traffic_event_ts) as latest_traffic,
        (SELECT MIN(kafka_timestamp) FROM parquet.`s3a://raw-data/openaq/`) as earliest_aq
    FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-----------------------+
|latest_traffic     |earliest_aq            |
+-------------------+-----------------------+
|2026-05-06 22:43:13|2026-05-06 22:58:50.598|
+-------------------+-----------------------+



In [9]:
spark.sql("""
    SELECT 
        traffic_borough,
        traffic_id,
        COUNT(*) as aq_matches_per_traffic_row
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough, traffic_id
    ORDER BY aq_matches_per_traffic_row DESC
    LIMIT 20
""").show(truncate=False)

+---------------+----------+--------------------------+
|traffic_borough|traffic_id|aq_matches_per_traffic_row|
+---------------+----------+--------------------------+
|Manhattan      |124       |277160                    |
|Brooklyn       |261       |216320                    |
|Brooklyn       |258       |209560                    |
|Manhattan      |150       |125060                    |
|Queens         |365       |104780                    |
|Manhattan      |445       |101400                    |
|Bronx          |191       |63600                     |
|Bronx          |195       |63600                     |
|Bronx          |177       |62275                     |
|Manhattan      |265       |60950                     |
|Bronx          |190       |60950                     |
|Staten Island  |385       |60840                     |
|Bronx          |186       |58300                     |
|Manhattan      |221       |10140                     |
|Manhattan      |215       |10140               

In [10]:
spark.sql("""
    SELECT 
        h3_index,
        COUNT(*) as rows
    FROM local.db.enriched_traffic
    GROUP BY h3_index
    ORDER BY rows DESC
    LIMIT 10
""").show(truncate=False)

+---------------+------+
|h3_index       |rows  |
+---------------+------+
|882a10755dfffff|425880|
|882a100ae3fffff|369675|
|882a107297fffff|277160|
|882a1072d7fffff|125060|
|882a100d01fffff|104780|
|882a1072c5fffff|101400|
|882a106231fffff|60840 |
|882a100d2bfffff|20280 |
|882a1072d9fffff|3380  |
+---------------+------+



In [11]:
spark.sql("""
SELECT
  MIN(traffic_event_ts) AS min_traffic,
  MAX(traffic_event_ts) AS max_traffic,
  MIN(aq_event_ts) AS min_aq,
  MAX(aq_event_ts) AS max_aq,
  MIN(weather_event_ts) AS min_weather,
  MAX(weather_event_ts) AS max_weather
FROM local.db.enriched_traffic
""").show(truncate=False)

spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_rows,
  SUM(CASE WHEN weather_temperature IS NOT NULL THEN 1 ELSE 0 END) AS weather_rows
FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-------------------+-------------------+-----------------------+-------------------+-------------------+
|min_traffic        |max_traffic        |min_aq             |max_aq                 |min_weather        |max_weather        |
+-------------------+-------------------+-------------------+-----------------------+-------------------+-------------------+
|2026-05-06 21:18:00|2026-05-06 22:43:13|2016-03-10 08:00:00|2026-05-06 23:36:57.372|2026-05-06 22:47:04|2026-05-06 23:34:27|
+-------------------+-------------------+-------------------+-----------------------+-------------------+-------------------+

+----------+-------+------------+
|total_rows|aq_rows|weather_rows|
+----------+-------+------------+
|1488455   |1488455|1488455     |
+----------+-------+------------+



In [12]:
# AQ Debugging

spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_matched_rows,
  ROUND(100.0 * SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS aq_match_pct
FROM local.db.enriched_traffic
""").show(truncate=False)

+----------+---------------+------------+
|total_rows|aq_matched_rows|aq_match_pct|
+----------+---------------+------------+
|1488455   |1488455        |100.00      |
+----------+---------------+------------+



In [13]:
spark.sql("""
SELECT
  MIN(traffic_event_ts) AS min_traffic, MAX(traffic_event_ts) AS max_traffic,
  MIN(aq_event_ts) AS min_aq, MAX(aq_event_ts) AS max_aq
FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-------------------+-------------------+-----------------------+
|min_traffic        |max_traffic        |min_aq             |max_aq                 |
+-------------------+-------------------+-------------------+-----------------------+
|2026-05-06 21:18:00|2026-05-06 22:43:13|2016-03-10 08:00:00|2026-05-06 23:36:57.372|
+-------------------+-------------------+-------------------+-----------------------+



In [14]:
spark.sql("""
SELECT
  traffic_borough,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_non_null_rows,
  ROUND(100.0 * SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS aq_non_null_pct
FROM local.db.enriched_traffic
GROUP BY traffic_borough
ORDER BY traffic_borough;
""").show(truncate=False)

+---------------+----------+----------------+---------------+
|traffic_borough|total_rows|aq_non_null_rows|aq_non_null_pct|
+---------------+----------+----------------+---------------+
|Bronx          |308725    |308725          |100.00         |
|Brooklyn       |425880    |425880          |100.00         |
|Manhattan      |588230    |588230          |100.00         |
|Queens         |104780    |104780          |100.00         |
|Staten Island  |60840     |60840           |100.00         |
+---------------+----------+----------------+---------------+



In [15]:
#Speed bucket -> PM2.5 distribution (the core feedback loop)
spark.sql("""
    SELECT
        CASE
            WHEN traffic_speed < 10 THEN '0-10 mph (gridlock)'
            WHEN traffic_speed < 25 THEN '10-25 mph (slow)'
            WHEN traffic_speed < 45 THEN '25-45 mph (moderate)'
            ELSE '45+ mph (free flow)'
        END AS speed_bucket,
        COUNT(*) AS records,
        ROUND(AVG(aq_pm25_ugm3), 2) AS avg_pm25,
        ROUND(STDDEV(aq_pm25_ugm3), 2) AS stddev_pm25
    FROM local.db.enriched_traffic
    WHERE aq_pm25_ugm3 IS NOT NULL
    GROUP BY 1
    ORDER BY avg_pm25 DESC
""").show(truncate=False)

+--------------------+-------+--------+-----------+
|speed_bucket        |records|avg_pm25|stddev_pm25|
+--------------------+-------+--------+-----------+
|0-10 mph (gridlock) |392940 |21.33   |6.03       |
|10-25 mph (slow)    |620075 |12.69   |6.97       |
|25-45 mph (moderate)|265880 |11.6    |5.95       |
|45+ mph (free flow) |209560 |9.72    |4.93       |
+--------------------+-------+--------+-----------+



In [16]:
# Congestion zone vs non-congestion zone PM2.5
spark.sql("""
    SELECT
        is_in_congestion_zone,
        is_near_truck_route,
        COUNT(*) AS records,
        ROUND(AVG(traffic_speed), 2) AS avg_speed,
        ROUND(AVG(aq_pm25_ugm3), 2) AS avg_pm25
    FROM local.db.enriched_traffic
    WHERE aq_pm25_ugm3 IS NOT NULL
    GROUP BY is_in_congestion_zone, is_near_truck_route
    ORDER BY avg_pm25 DESC
""").show(truncate=False)

+---------------------+-------------------+-------+---------+--------+
|is_in_congestion_zone|is_near_truck_route|records|avg_speed|avg_pm25|
+---------------------+-------------------+-------+---------+--------+
|false                |true               |1464795|21.74    |14.4    |
|false                |false              |23660  |14.04    |11.78   |
+---------------------+-------------------+-------+---------+--------+



In [17]:
#Hour-of-day pattern — rush hour signal
spark.sql("""
    SELECT
        HOUR(traffic_event_ts) AS hour_of_day,
        ROUND(AVG(traffic_speed), 2) AS avg_speed,
        ROUND(AVG(aq_pm25_ugm3), 2) AS avg_pm25,
        COUNT(*) AS records
    FROM local.db.enriched_traffic
    WHERE aq_pm25_ugm3 IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").show(24, truncate=False)

+-----------+---------+--------+-------+
|hour_of_day|avg_speed|avg_pm25|records|
+-----------+---------+--------+-------+
|21         |20.93    |14.15   |506720 |
|22         |21.97    |14.46   |981735 |
+-----------+---------+--------+-------+



In [18]:
# Wind speed as confounder - Does wind dilute the PM2.5 readings and reduce the correlation with traffic speed?
spark.sql("""
    SELECT
        CASE
            WHEN weather_wind_speed_mph < 5  THEN 'Calm (<5 mph)'
            WHEN weather_wind_speed_mph < 15 THEN 'Breezy (5-15 mph)'
            ELSE 'Windy (15+ mph)'
        END AS wind_category,
        ROUND(AVG(aq_pm25_ugm3), 2) AS avg_pm25,
        ROUND(AVG(traffic_speed), 2) AS avg_speed,
        COUNT(*) AS records
    FROM local.db.enriched_traffic
    WHERE aq_pm25_ugm3 IS NOT NULL AND weather_wind_speed_mph IS NOT NULL
    GROUP BY 1
    ORDER BY avg_pm25 DESC
""").show(truncate=False)

+-----------------+--------+---------+-------+
|wind_category    |avg_pm25|avg_speed|records|
+-----------------+--------+---------+-------+
|Breezy (5-15 mph)|14.36   |21.62    |1488455|
+-----------------+--------+---------+-------+



In [19]:
# Top 10 most polluted H3 cells with low traffic speed
spark.sql("""
    SELECT
        h3_index,
        traffic_borough,
        ROUND(AVG(traffic_speed), 2) AS avg_speed,
        ROUND(AVG(aq_pm25_ugm3), 2) AS avg_pm25,
        COUNT(*) AS records,
        CAST(MAX(is_near_truck_route) AS STRING) AS on_truck_route
    FROM local.db.enriched_traffic
    WHERE aq_pm25_ugm3 IS NOT NULL AND traffic_speed < 15
    GROUP BY h3_index, traffic_borough
    ORDER BY avg_pm25 DESC
    LIMIT 10
""").show(truncate=False)

+---------------+---------------+---------+--------+-------+--------------+
|h3_index       |traffic_borough|avg_speed|avg_pm25|records|on_truck_route|
+---------------+---------------+---------+--------+-------+--------------+
|882a1072d7fffff|Manhattan      |2.75     |29.92   |125060 |true          |
|882a1072d9fffff|Manhattan      |9.84     |19.89   |3380   |false         |
|882a100ae3fffff|Bronx          |8.74     |17.6    |241445 |true          |
|882a100ae3fffff|Manhattan      |10.39    |17.6    |34960  |true          |
|882a107297fffff|Manhattan      |9.89     |17.4    |277160 |true          |
|882a100d2bfffff|Manhattan      |9.84     |10.43   |10140  |false         |
|882a1072c5fffff|Manhattan      |11.81    |4.97    |58800  |true          |
|882a100d01fffff|Queens         |12.6     |1.38    |104780 |true          |
+---------------+---------------+---------+--------+-------+--------------+

